# MIP Notebook Client - Getting Started

This notebook shows how to discover the `mip` facade API and run experiments.

# 5. Metadata hierarchy tree
print(metadata.describe('dementia:0.1', max_lines=80))

# Include variable leaves (usually larger output)
print(metadata.describe('dementia:0.1', include_variables=True, max_lines=120))

In [ ]:
import inspect
import mip
from mip import configure, experiments, metadata, algorithms

# 1. Setup
# No manual token needed when running via /notebook (PORTAL_TOKEN is injected).
# If running outside docker-compose, you likely need:
#   configure(base_url='http://localhost:8080/services', token='...')
configure()
print('Facade exports:', ', '.join(mip.__all__))

In [ ]:
# 2. Discover the facade API
def show_module_api(module, name):
    print(f"\n{name}")
    for member_name, member in inspect.getmembers(module):
        if member_name.startswith('_'):
            continue
        if callable(member):
            try:
                signature = inspect.signature(member)
            except (TypeError, ValueError):
                signature = '(...)'
            print(f"  - {name}.{member_name}{signature}")

for module_name, module in [('experiments', experiments), ('metadata', metadata), ('algorithms', algorithms)]:
    show_module_api(module, module_name)

print('\nTip: run help(experiments.create) or help(metadata.get_pathology).')

In [ ]:
# 3. List recent experiments (if any)
recent_experiments = experiments.list(limit=20)
print(f'Found {len(recent_experiments)} experiments.')
for exp in recent_experiments[:5]:
    print(f'- {exp.name} ({exp.status})')

In [ ]:
# 4. Run a transient experiment (returns results immediately; not saved in history)
# Example (adjust to your environment / available algorithms and data models)
transient = experiments.run_transient(
    name='Quick Check',
    algorithm_name='descriptive_stats',
    data_model='dementia:0.1',
    datasets=['edsd'],
    y=['alzheimer_broad_category', 'gender'],
    x=[],
)
print('Status:', transient.status)
transient.results

# If you want a persisted experiment (shows up in history), use:
#   exp = experiments.create(...)
#   exp.wait(timeout=600)